In [ ]:
import time
import random
import csv
import os
import re

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait


# ==============================
# CONFIG
# ==============================

BASE_URL = "https://www.indiabix.com/data-interpretation/questions-and-answers/"
CSV_FILE = "indiabixchartsdataset.csv"

MIN_DELAY = 2
MAX_DELAY = 4

VALID_CHART_TYPES = [
    "bar charts",
    "line charts",
    "pie charts",
    "mixed graphs",
    "radar charts"
]


# ==============================
# DELAYS
# ==============================

def human_delay():
    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))


def small_delay():
    time.sleep(random.uniform(1, 2))


# ==============================
# DRIVER
# ==============================

def create_driver():

    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")

    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(options=options)

    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )

    return driver


# ==============================
# CSV SETUP
# ==============================

def setup_csv():

    with open(CSV_FILE, "w", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)

        writer.writerow([
            "chart_type",
            "page_number",
            "chart_image_url",
            "question",
            "option_a",
            "option_b",
            "option_c",
            "option_d",
            "correct_answer"
        ])


def save_to_csv(data):

    with open(CSV_FILE, "a", newline="", encoding="utf-8", errors="ignore") as f:

        writer = csv.writer(f)
        writer.writerow(data)


# ==============================
# GET CHART IMAGE
# ==============================

def get_chart_image(driver):

    try:

        images = driver.find_elements(By.TAG_NAME, "img")

        for img in images:

            src = img.get_attribute("src")

            if src and (
                "chart" in src.lower()
                or "graph" in src.lower()
                or "diagram" in src.lower()
            ):
                return src

    except:
        pass

    return ""


# ==============================
# PAGINATION
# ==============================

def go_to_next_page(driver):

    try:

        next_li = driver.find_element(
            By.XPATH,
            "//ul[contains(@class,'pagination')]//li[.//a[normalize-space()='Next']]"
        )

        if "disabled" in next_li.get_attribute("class").lower():
            return False

        next_btn = next_li.find_element(By.TAG_NAME, "a")

        driver.execute_script("arguments[0].click();", next_btn)

        human_delay()

        return True

    except:
        return False


# ==============================
# SCRAPE QUESTIONS
# ==============================

def scrape_exercise(driver, chart_type):

    wait = WebDriverWait(driver, 10)

    page_number = 1

    while True:

        human_delay()

        chart_image = get_chart_image(driver)

        question_blocks = wait.until(
            lambda d: d.find_elements(By.CLASS_NAME, "bix-div-container")
        )

        print(f"{chart_type} | Page {page_number} → {len(question_blocks)} questions")

        for block in question_blocks:

            try:
                question = block.find_element(
                    By.CLASS_NAME,
                    "bix-td-qtxt"
                ).text.strip()
            except:
                continue

            option_rows = block.find_elements(By.CLASS_NAME, "bix-opt-row")

            option_texts = []
            correct_answer = ""

            # collect options
            for row in option_rows:

                try:
                    val = row.find_element(By.CLASS_NAME, "bix-td-option-val")
                    option_texts.append(val.text.strip())
                except:
                    option_texts.append("")

            while len(option_texts) < 4:
                option_texts.append("")

            # click options to reveal answer
            for row in option_rows:

                try:
                    btn = row.find_element(By.CLASS_NAME, "bix-td-option")
                    driver.execute_script("arguments[0].click();", btn)
                    small_delay()
                except:
                    continue

            time.sleep(1)

            for row in option_rows:

                try:

                    val = row.find_element(By.CLASS_NAME, "bix-td-option-val")

                    classes = val.get_attribute("class") or ""

                    if "correct" in classes.lower() or "right" in classes.lower():

                        correct_answer = val.text.strip()
                        break

                except:
                    continue

            save_to_csv([
                chart_type,
                page_number,
                chart_image,
                question,
                option_texts[0],
                option_texts[1],
                option_texts[2],
                option_texts[3],
                correct_answer
            ])

        moved = go_to_next_page(driver)

        if not moved:
            break

        page_number += 1


# ==============================
# MAIN
# ==============================

def main():

    setup_csv()

    driver = create_driver()

    wait = WebDriverWait(driver, 15)

    print("Loading Data Interpretation index...")

    driver.get(BASE_URL)

    human_delay()

    chart_links = wait.until(
        lambda d: d.find_elements(By.XPATH, "//ul[@class='need-ul-filter']//a")
    )

    chart_types = {}

    for link in chart_links:

        name = link.text.strip().lower()
        url = link.get_attribute("href")

        if any(t in name for t in VALID_CHART_TYPES):

            chart_types[name.title()] = url
            print("Found:", name.title())

    print("\nStarting scraping...")

    for chart_type, chart_url in chart_types.items():

        print("\n" + "="*60)
        print("PROCESSING:", chart_type)
        print("="*60)

        driver.get(chart_url)

        human_delay()

        exercise_links = driver.find_elements(
            By.XPATH,
            "//a[contains(@href,'/data-interpretation/') and string-length(normalize-space(text())) > 0]"
        )

        exercises = []

        for e in exercise_links:

            name = e.text.strip()
            url = e.get_attribute("href")

            if name:
                nums = re.findall(r"\d+", name)

                if nums:
                    exercises.append((int(nums[0]), url))

        exercises.sort(key=lambda x: x[0])

        exercises = [url for _, url in exercises]

        print("Exercises found:", len(exercises))

        for ex in exercises:

            driver.get(ex)
            human_delay()

            scrape_exercise(driver, chart_type)

    driver.quit()

    print("\nSCRAPING COMPLETED")
    print("Saved at:", os.path.abspath(CSV_FILE))


if __name__ == "__main__":
    main()

In [1]:
import time
import random
import csv
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait


# ==============================
# CONFIG
# ==============================

BASE_URL = "https://www.indiabix.com/data-interpretation/bar-charts/"
CSV_FILE = "bar_chart_dataset.csv"

SECTION = "Data Interpretation"
TOPIC = "Bar Charts"

MIN_DELAY = 2
MAX_DELAY = 4


# ==============================
# DELAYS
# ==============================

def human_delay():
    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))


def small_delay():
    time.sleep(random.uniform(1, 2))


# ==============================
# DRIVER
# ==============================

def create_driver():

    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(options=options)

    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )

    return driver


# ==============================
# CSV SETUP
# ==============================

def setup_csv():

    with open(CSV_FILE, "w", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)

        writer.writerow([
            "Section",
            "TopicName",
            "PageNumber",
            "ImageURL",
            "Question",
            "Option_A",
            "Option_B",
            "Option_C",
            "Option_D",
            "CorrectAnswer"
        ])


def save_to_csv(data):

    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)
        writer.writerow(data)


# ==============================
# GET CHART IMAGE
# ==============================

def get_chart_image(driver):

    try:

        images = driver.find_elements(By.TAG_NAME, "img")

        for img in images:

            src = img.get_attribute("src")

            if src and ("chart" in src.lower() or "graph" in src.lower()):
                return src

    except:
        pass

    return ""


# ==============================
# NEXT PAGE
# ==============================

def go_to_next_page(driver):

    try:

        next_btn = driver.find_element(
            By.XPATH,
            "//ul[contains(@class,'pagination')]//a[normalize-space()='Next']"
        )

        driver.execute_script("arguments[0].click();", next_btn)

        human_delay()

        return True

    except:
        return False


# ==============================
# SCRAPE QUESTIONS
# ==============================

def scrape_questions(driver):

    wait = WebDriverWait(driver, 10)

    page_number = 1

    while True:

        human_delay()

        chart_image = get_chart_image(driver)

        question_blocks = wait.until(
            lambda d: d.find_elements(By.CLASS_NAME, "bix-div-container")
        )

        print(f"Bar Charts | Page {page_number} → {len(question_blocks)} questions")

        for block in question_blocks:

            try:
                question = block.find_element(By.CLASS_NAME, "bix-td-qtxt").text.strip()
            except:
                continue

            option_rows = block.find_elements(By.CLASS_NAME, "bix-opt-row")

            option_texts = []
            correct_answer = ""

            for row in option_rows:

                try:
                    val = row.find_element(By.CLASS_NAME, "bix-td-option-val")
                    option_texts.append(val.text.strip())
                except:
                    option_texts.append("")

            while len(option_texts) < 4:
                option_texts.append("")

            # click options to reveal answer
            for row in option_rows:

                try:
                    btn = row.find_element(By.CLASS_NAME, "bix-td-option")
                    driver.execute_script("arguments[0].click();", btn)
                    small_delay()
                except:
                    continue

            time.sleep(1)

            for row in option_rows:

                try:

                    val = row.find_element(By.CLASS_NAME, "bix-td-option-val")
                    classes = val.get_attribute("class") or ""

                    if "correct" in classes.lower() or "right" in classes.lower():
                        correct_answer = val.text.strip()
                        break

                except:
                    continue

            save_to_csv([
                SECTION,
                TOPIC,
                page_number,
                chart_image,
                question,
                option_texts[0],
                option_texts[1],
                option_texts[2],
                option_texts[3],
                correct_answer
            ])

        if not go_to_next_page(driver):
            break

        page_number += 1


# ==============================
# MAIN
# ==============================

def main():

    setup_csv()

    driver = create_driver()

    print("Opening Bar Charts page...")

    driver.get(BASE_URL)

    human_delay()

    scrape_questions(driver)

    driver.quit()

    print("\nScraping finished")
    print("CSV saved at:", os.path.abspath(CSV_FILE))


if __name__ == "__main__":
    main()

Opening Bar Charts page...
Bar Charts | Page 1 → 5 questions

Scraping finished
CSV saved at: C:\Users\nandh\bar_chart_dataset.csv


In [1]:
import time
import random
import csv
import os
import re

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait


# ==============================
# CONFIG
# ==============================

BASE_URL = "https://www.indiabix.com/data-interpretation/questions-and-answers/"
CSV_FILE = "INDIABIXCHARTSDATASET.csv"

SECTION = "Data Interpretation"

MIN_DELAY = 2
MAX_DELAY = 4

VALID_CHART_TYPES = [
    "bar charts",
    "line charts",
    "pie charts",
    "mixed graphs",
    "radar charts"
]


# ==============================
# DELAYS
# ==============================

def human_delay():
    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))


def small_delay():
    time.sleep(random.uniform(1, 2))


# ==============================
# DRIVER
# ==============================

def create_driver():

    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")

    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(options=options)

    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )

    return driver


# ==============================
# CSV SETUP
# ==============================

def setup_csv():

    with open(CSV_FILE, "w", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)

        writer.writerow([
            "Section",
            "TopicName",
            "PageNumber",
            "ImageURL",
            "Question",
            "Option_A",
            "Option_B",
            "Option_C",
            "Option_D",
            "CorrectAnswer"
        ])


def save_to_csv(data):

    with open(CSV_FILE, "a", newline="", encoding="utf-8", errors="ignore") as f:

        writer = csv.writer(f)
        writer.writerow(data)


# ==============================
# GET CHART IMAGE
# ==============================

def get_chart_image(driver):

    try:

        images = driver.find_elements(By.TAG_NAME, "img")

        for img in images:

            src = img.get_attribute("src")

            if src and (
                "chart" in src.lower()
                or "graph" in src.lower()
                or "diagram" in src.lower()
            ):
                return src

    except:
        pass

    return ""


# ==============================
# PAGINATION
# ==============================

def go_to_next_page(driver):

    try:

        next_li = driver.find_element(
            By.XPATH,
            "//ul[contains(@class,'pagination')]//li[.//a[normalize-space()='Next']]"
        )

        if "disabled" in next_li.get_attribute("class").lower():
            return False

        next_btn = next_li.find_element(By.TAG_NAME, "a")

        driver.execute_script("arguments[0].click();", next_btn)

        human_delay()

        return True

    except:
        return False


# ==============================
# SCRAPE QUESTIONS
# ==============================

def scrape_exercise(driver, chart_type):

    wait = WebDriverWait(driver, 10)

    page_number = 1

    while True:

        human_delay()

        chart_image = get_chart_image(driver)

        question_blocks = wait.until(
            lambda d: d.find_elements(By.CLASS_NAME, "bix-div-container")
        )

        print(f"{chart_type} | Page {page_number} → {len(question_blocks)} questions")

        for block in question_blocks:

            try:
                question = block.find_element(
                    By.CLASS_NAME,
                    "bix-td-qtxt"
                ).text.strip()
            except:
                continue

            option_rows = block.find_elements(By.CLASS_NAME, "bix-opt-row")

            option_texts = []
            correct_answer = ""

            # collect options
            for row in option_rows:

                try:
                    val = row.find_element(By.CLASS_NAME, "bix-td-option-val")
                    option_texts.append(val.text.strip())
                except:
                    option_texts.append("")

            while len(option_texts) < 4:
                option_texts.append("")

            # click options to reveal answer
            for row in option_rows:

                try:
                    btn = row.find_element(By.CLASS_NAME, "bix-td-option")
                    driver.execute_script("arguments[0].click();", btn)
                    small_delay()
                except:
                    continue

            time.sleep(1)

            for row in option_rows:

                try:

                    val = row.find_element(By.CLASS_NAME, "bix-td-option-val")

                    classes = val.get_attribute("class") or ""

                    if "correct" in classes.lower() or "right" in classes.lower():

                        correct_answer = val.text.strip()
                        break

                except:
                    continue

            save_to_csv([
                SECTION,
                chart_type,
                page_number,
                chart_image,
                question,
                option_texts[0],
                option_texts[1],
                option_texts[2],
                option_texts[3],
                correct_answer
            ])

        moved = go_to_next_page(driver)

        if not moved:
            break

        page_number += 1


# ==============================
# MAIN
# ==============================

def main():

    setup_csv()

    driver = create_driver()

    wait = WebDriverWait(driver, 15)

    print("Loading Data Interpretation index...")

    driver.get(BASE_URL)

    human_delay()

    chart_links = wait.until(
        lambda d: d.find_elements(By.XPATH, "//ul[@class='need-ul-filter']//a")
    )

    chart_types = {}

    for link in chart_links:

        name = link.text.strip().lower()
        url = link.get_attribute("href")

        if any(t in name for t in VALID_CHART_TYPES):

            chart_types[name.title()] = url
            print("Found:", name.title())

    print("\nStarting scraping...")

    for chart_type, chart_url in chart_types.items():

        print("\n" + "="*60)
        print("PROCESSING:", chart_type)
        print("="*60)

        driver.get(chart_url)

        human_delay()

        exercise_links = driver.find_elements(
            By.XPATH,
            "//a[contains(@href,'/data-interpretation/') and string-length(normalize-space(text())) > 0]"
        )

        exercises = []

        for e in exercise_links:

            name = e.text.strip()
            url = e.get_attribute("href")

            if name:
                nums = re.findall(r"\d+", name)

                if nums:
                    exercises.append((int(nums[0]), url))

        exercises.sort(key=lambda x: x[0])

        exercises = [url for _, url in exercises]

        print("Exercises found:", len(exercises))

        for ex in exercises:

            driver.get(ex)
            human_delay()

            scrape_exercise(driver, chart_type)

    driver.quit()

    print("\nSCRAPING COMPLETED")
    print("Saved at:", os.path.abspath(CSV_FILE))


if __name__ == "__main__":
    main()

Loading Data Interpretation index...
Found: Bar Charts
Found: Pie Charts
Found: Line Charts

Starting scraping...

PROCESSING: Bar Charts
Exercises found: 19
Bar Charts | Page 1 → 5 questions
Bar Charts | Page 1 → 5 questions
Bar Charts | Page 1 → 5 questions
Bar Charts | Page 2 → 1 questions
Bar Charts | Page 1 → 5 questions
Bar Charts | Page 1 → 5 questions
Bar Charts | Page 1 → 5 questions
Bar Charts | Page 1 → 5 questions
Bar Charts | Page 1 → 5 questions
Bar Charts | Page 1 → 5 questions
Bar Charts | Page 1 → 5 questions
Bar Charts | Page 1 → 4 questions
Bar Charts | Page 1 → 5 questions
Bar Charts | Page 1 → 4 questions
Bar Charts | Page 1 → 4 questions
Bar Charts | Page 1 → 5 questions
Bar Charts | Page 1 → 3 questions
Bar Charts | Page 1 → 5 questions
Bar Charts | Page 1 → 5 questions
Bar Charts | Page 1 → 5 questions

PROCESSING: Pie Charts
Exercises found: 13
Pie Charts | Page 1 → 5 questions
Pie Charts | Page 2 → 4 questions
Pie Charts | Page 1 → 3 questions
Pie Charts | Pag